# 🔄 ETL Pipeline — Proyecto Olist

> **Qué hace este notebook:**
> Extrae los CSVs del dataset de Olist, los limpia y transforma con Pandas, y los carga en PostgreSQL.
>
> **Cuándo ejecutarlo:**
> Cada vez que quieras resetear los datos en PostgreSQL. Ejecuta **Kernel → Restart & Run All**.

---

## El flujo completo

```
CSVs (data/raw/)
      ↓
[EXTRACT]  Leer ficheros con Pandas
      ↓
[TRANSFORM] Limpiar, convertir tipos, crear columnas nuevas
      ↓
[LOAD]     Cargar en PostgreSQL
      ↓
Listo para consultar en 02_sql_queries.ipynb
```

---
## ⚙️ Setup

Imports y conexión a PostgreSQL. **Siempre se ejecuta primero.**

In [1]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

# Rutas del proyecto
ROOT = Path().resolve().parent
DATA_PATH = ROOT / "data" / "raw"

# Conexión a PostgreSQL
# Cambia 'tu_usuario' por tu usuario de Mac (comando: whoami)
engine = create_engine("postgresql://tomas@localhost:5432/data_engineering")

print("✅ Setup completado")
print(f"📁 Ruta de datos: {DATA_PATH}")

✅ Setup completado
📁 Ruta de datos: /Users/tomas/Data/data-engineering-project/data/raw


---
## 1. EXTRACT — Leer los CSVs

En esta fase simplemente leemos los ficheros tal como están, sin modificar nada.
El objetivo es tener los datos en memoria para poder trabajar con ellos.

> 💡 **Regla importante:** nunca modifiques los ficheros de `data/raw/`.
> Son los datos originales y deben mantenerse intactos siempre.

In [2]:
# Leer todos los CSVs del dataset
orders_raw    = pd.read_csv(DATA_PATH / "olist_orders_dataset.csv")
customers_raw = pd.read_csv(DATA_PATH / "olist_customers_dataset.csv")
payments_raw  = pd.read_csv(DATA_PATH / "olist_order_payments_dataset.csv")

print("✅ Extracción completada")
print(f"   orders:    {orders_raw.shape[0]:,} filas, {orders_raw.shape[1]} columnas")
print(f"   customers: {customers_raw.shape[0]:,} filas, {customers_raw.shape[1]} columnas")
print(f"   payments:  {payments_raw.shape[0]:,} filas, {payments_raw.shape[1]} columnas")

✅ Extracción completada
   orders:    99,441 filas, 8 columnas
   customers: 99,441 filas, 5 columnas
   payments:  103,886 filas, 5 columnas


---
## 2. TRANSFORM — Limpiar y transformar

Aquí es donde limpiamos los datos antes de cargarlos.
Trabajaremos con copias para no modificar los datos originales en memoria.

**Transformaciones que aplicamos:**
- Convertir columnas de fecha de `object` a `datetime`
- Filtrar solo pedidos entregados (`order_status = 'delivered'`)
- Crear columna `delivery_days` (días entre compra y entrega)

In [3]:
# --- TRANSFORM: orders ---

# Columnas que contienen fechas pero están como texto (object)
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

# Trabajamos con una copia para no modificar orders_raw
orders = orders_raw.copy()

# Convertir fechas de texto a datetime
# Sin esto no podemos hacer operaciones de fecha (calcular días, filtrar por mes, etc.)
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# Filtrar solo pedidos entregados
# Los cancelados, en camino, etc. no tienen fecha de entrega real
orders_clean = orders[orders["order_status"] == "delivered"].copy()

# Crear columna nueva: días que tardó cada pedido en entregarse
# Restamos dos fechas → resultado en días con .dt.days
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"] -
    orders_clean["order_purchase_timestamp"]
).dt.days

print("✅ Transform orders completado")
print(f"   Filas originales:  {orders_raw.shape[0]:,}")
print(f"   Filas después de filtrar delivered: {orders_clean.shape[0]:,}")
print(f"   Columnas: {list(orders_clean.columns)}")

✅ Transform orders completado
   Filas originales:  99,441
   Filas después de filtrar delivered: 96,478
   Columnas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days']


In [4]:
# --- TRANSFORM: customers ---
# No necesita transformaciones especiales, está bastante limpio
customers_clean = customers_raw.copy()

print("✅ Transform customers completado")
print(f"   Filas: {customers_clean.shape[0]:,}")
print(f"   Nulos:\n{customers_clean.isnull().sum()}")

✅ Transform customers completado
   Filas: 99,441
   Nulos:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [5]:
# --- TRANSFORM: payments ---
# No necesita transformaciones especiales
payments_clean = payments_raw.copy()

print("✅ Transform payments completado")
print(f"   Filas: {payments_clean.shape[0]:,}")
print(f"   Tipos de pago únicos: {payments_clean['payment_type'].unique()}")

✅ Transform payments completado
   Filas: 103,886
   Tipos de pago únicos: ['credit_card' 'boleto' 'voucher' 'debit_card' 'not_defined']


---
## 3. LOAD — Cargar en PostgreSQL

Cargamos los datos limpios en PostgreSQL.
A partir de aquí se pueden consultar desde `02_sql_queries.ipynb`.

**Parámetros importantes de `to_sql`:**
- `if_exists='replace'` → borra y recrea la tabla cada vez (útil en desarrollo)
- `index=False` → no guarda el índice numérico de Pandas como columna
- `chunksize=1000` → inserta de 1000 en 1000 filas (más eficiente)

In [6]:
# Cargar orders
orders_clean.to_sql(
    name="orders",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print(f"✅ Tabla 'orders' cargada: {orders_clean.shape[0]:,} filas")

# Cargar customers
customers_clean.to_sql(
    name="customers",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print(f"✅ Tabla 'customers' cargada: {customers_clean.shape[0]:,} filas")

# Cargar payments
payments_clean.to_sql(
    name="payments",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print(f"✅ Tabla 'payments' cargada: {payments_clean.shape[0]:,} filas")

✅ Tabla 'orders' cargada: 96,478 filas
✅ Tabla 'customers' cargada: 99,441 filas
✅ Tabla 'payments' cargada: 103,886 filas


---
## 4. Verificación

Comprobamos que todo se ha cargado correctamente.

In [7]:
# Verificar que las tablas existen y tienen el número correcto de filas
with engine.connect() as conn:
    for tabla in ["orders", "customers", "payments"]:
        result = conn.execute(__import__('sqlalchemy').text(f"SELECT COUNT(*) FROM {tabla}"))
        count = result.fetchone()[0]
        print(f"✅ {tabla}: {count:,} filas en PostgreSQL")

✅ orders: 96,478 filas en PostgreSQL
✅ customers: 99,441 filas en PostgreSQL
✅ payments: 103,886 filas en PostgreSQL


---
## ✅ Pipeline completado

Ya puedes abrir `02_sql_queries.ipynb` para consultar los datos con SQL.

**Tablas disponibles en PostgreSQL:**
| Tabla | Descripción |
|-------|-------------|
| `orders` | Pedidos entregados con días de entrega calculados |
| `customers` | Datos de clientes |
| `payments` | Pagos por pedido |